In [33]:
import pandas as pd
import random
from faker import Faker
from datetime import datetime, timedelta

In [34]:
fake = Faker()

customers = pd.DataFrame([
    {
        "customer_id": f"C{str(i).zfill(3)}",
        "name": fake.name(),
        "email": fake.email(),
        "city": fake.city(),
        "signup_date": fake.date_between(start_date='-2y', end_date='today')
    }
    for i in range(1, 9)
])

In [35]:
genres = ["Action", "Drama", "Comedy", "Sci-Fi", "Thriller", "Romance", "Animation"]

movies = pd.DataFrame([
    {
        "movie_id": f"M{str(i).zfill(3)}",
        "title": fake.catch_phrase(),
        "genre": random.choice(genres),
        "release_year": random.randint(1990, 2024),
        "rating": random.randint(1, 5)
    }
    for i in range(1, 11)
])

In [36]:
rentals_list = []
for i in range(1, 26):
    rental_date = fake.date_between(start_date='-1y', end_date='today')
    return_date = rental_date + timedelta(days=random.randint(1, 14))
    
    rentals_list.append({
        "rental_id": f"R{str(i).zfill(3)}",
        "customer_id": random.choice(customers["customer_id"]),
        "movie_id": random.choice(movies["movie_id"]),
        "rental_date": rental_date,
        "return_date": return_date,
        "price": round(random.uniform(3.0, 12.0), 2)
    })
rentals = pd.DataFrame(rentals_list)
rentals

,rental_id,customer_id,movie_id,rental_date,return_date,price
0,R001,C001,M007,2025-08-28,2025-09-02,6.64
1,R002,C006,M001,2025-10-11,2025-10-22,8.72
2,R003,C004,M008,2025-07-20,2025-07-28,3.35
3,R004,C007,M009,2025-12-21,2025-12-28,9.62
4,R005,C008,M009,2025-04-01,2025-04-04,6.13
5,R006,C001,M002,2025-04-28,2025-05-09,11.69
6,R007,C007,M008,2025-10-28,2025-11-02,7.34
7,R008,C004,M001,2026-02-04,2026-02-06,6.18
8,R009,C006,M005,2025-06-18,2025-06-19,3.65
9,R010,C002,M007,2025-11-05,2025-11-07,3.26


In [37]:
specific_customer = "C001"
customer_movies = (

    rentals[rentals["customer_id"] == specific_customer]

    .merge(movies, on="movie_id")

)


print("Movies rented by", specific_customer)

print(customer_movies[["title", "genre", "rental_date", "price"]])


Movies rented by C001
                                       title     genre rental_date  price
0                  Upgradable static ability  Thriller  2025-08-28   6.64
1            Streamlined logistical alliance    Sci-Fi  2025-04-28  11.69
2  Reverse-engineered interactive initiative  Thriller  2025-10-15  10.34
3                  Upgradable static ability  Thriller  2025-05-10   3.44
4              Upgradable analyzing paradigm    Comedy  2026-01-25   8.99
5            Re-engineered impactful support     Drama  2025-09-05   6.73
6               Devolved full-range software   Romance  2025-05-04  10.06


In [38]:
top_5_movies = (

    rentals.groupby("movie_id")

    .size()

    .reset_index(name="rental_count")

    .merge(movies[["movie_id", "title"]], on="movie_id")

    .sort_values("rental_count", ascending=False)

    .head(5)

)


In [39]:
print("\n2) Top 5 Most-Rented Movies")

print(top_5_movies)





2) Top 5 Most-Rented Movies
  movie_id  rental_count                                   title
5     M007             6               Upgradable static ability
0     M001             4           Upgradable analyzing paradigm
7     M009             4            Devolved full-range software
6     M008             3  Reverse-engineered national complexity
1     M002             2         Streamlined logistical alliance


In [40]:
revenue_by_genre = (

    rentals.merge(movies, on="movie_id")

    .groupby("genre")["price"]

    .sum()

    .reset_index()

    .sort_values("price", ascending=False)

)



print("\n3) Total Revenue Per Genre")

print(revenue_by_genre)





3) Total Revenue Per Genre
       genre  price
5   Thriller  47.78
1     Comedy  43.12
3    Romance  29.63
0  Animation  20.67
2      Drama  15.62
4     Sci-Fi  14.78


In [41]:
low_rated_ids = movies[movies["rating"] < 3]["movie_id"]



customers_with_low = rentals[rentals["movie_id"].isin(low_rated_ids)]["customer_id"].unique()



eligible_customers = customers[~customers["customer_id"].isin(customers_with_low)]



print("\n4) Customers Who Never Rented Movie Rated Below 3")

print(eligible_customers)



4) Customers Who Never Rented Movie Rated Below 3
  customer_id                name                   email             city  \
1        C002          Peter Gill  michelle14@example.net     Port Brandon   
2        C003  Christopher Hughes    ashley97@example.com  West Sandraberg   
4        C005          Jamie Hill     bprince@example.com     Davidborough   
5        C006     Jose Mccullough      klucas@example.com       Austinberg   

  signup_date  
1  2025-11-23  
2  2024-09-19  
4  2024-11-13  
5  2025-02-10  


In [42]:
denormalized = rentals.merge(customers, on="customer_id").merge(movies, on="movie_id")


In the denormalized table, customer information (name, email, city) and movie information (title, genre, rating) are repeated for every rental. If one customer rents 5 movies, their personal data appears 5 times. If one movie is rented 10 times, its information appears 10 times. This causes redundancy and update anomalies. For example, if a customer changes their email, we must update many rows. If we miss one row, the database becomes inconsistent. The normalized design avoids this by separating: customers, movies, and rentals into independent tables and connecting them using foreign keys. This reduces redundancy and makes updates safer and cleaner.


In [43]:
movie_lookup = movies.set_index("movie_id").to_dict(orient="index")

customer_documents = []

for _, cust in customers.iterrows():
    cust_rentals = rentals[rentals["customer_id"] == cust["customer_id"]]
    embedded_rentals = []  
    for _, r in cust_rentals.iterrows():
        movie_data = movie_lookup[r["movie_id"]]       
        embedded_rentals.append({
            "rental_id": r["rental_id"],
            "rental_date": r["rental_date"],
            "return_date": r["return_date"],
            "price": r["price"],
            "movie": movie_data
        })   

    customer_documents.append({
        "customer_id": cust["customer_id"],
        "name": cust["name"],
        "email": cust["email"],
        "city": cust["city"],
        "signup_date": cust["signup_date"],
        "rentals": embedded_rentals

    })


print("Sample Document:\n")
print(customer_documents[0])


Sample Document:

{'customer_id': 'C001', 'name': 'Adam Kim', 'email': 'pbranch@example.net', 'city': 'Port Brittney', 'signup_date': datetime.date(2024, 5, 11), 'rentals': [{'rental_id': 'R001', 'rental_date': datetime.date(2025, 8, 28), 'return_date': datetime.date(2025, 9, 2), 'price': 6.64, 'movie': {'title': 'Upgradable static ability', 'genre': 'Thriller', 'release_year': 2008, 'rating': 4}}, {'rental_id': 'R006', 'rental_date': datetime.date(2025, 4, 28), 'return_date': datetime.date(2025, 5, 9), 'price': 11.69, 'movie': {'title': 'Streamlined logistical alliance', 'genre': 'Sci-Fi', 'release_year': 2024, 'rating': 3}}, {'rental_id': 'R011', 'rental_date': datetime.date(2025, 10, 15), 'return_date': datetime.date(2025, 10, 29), 'price': 10.34, 'movie': {'title': 'Reverse-engineered interactive initiative', 'genre': 'Thriller', 'release_year': 2023, 'rating': 2}}, {'rental_id': 'R020', 'rental_date': datetime.date(2025, 5, 10), 'return_date': datetime.date(2025, 5, 22), 'price': 

In [44]:
doc_movies = [rental["movie"]["title"]
    for customer in customer_documents
    if customer["customer_id"] == specific_customer
    for rental in customer["rentals"]
]
print("1) Movies rented by", specific_customer)
print(doc_movies)

1) Movies rented by C001
['Upgradable static ability', 'Streamlined logistical alliance', 'Reverse-engineered interactive initiative', 'Upgradable static ability', 'Upgradable analyzing paradigm', 'Re-engineered impactful support', 'Devolved full-range software']


In [45]:
movie_count = {}
for customer in customer_documents:
    for rental in customer["rentals"]:
        title = rental["movie"]["title"]
        movie_count[title] = movie_count.get(title, 0) + 1
top_5_doc = sorted(movie_count.items(), key=lambda x: x[1], reverse=True)[:5]


print("\n2) Top 5 Most-Rented Movies")
print(top_5_doc)


2) Top 5 Most-Rented Movies
[('Upgradable static ability', 6), ('Upgradable analyzing paradigm', 4), ('Devolved full-range software', 4), ('Reverse-engineered national complexity', 3), ('Streamlined logistical alliance', 2)]


In [46]:
revenue_doc = {}
for customer in customer_documents:
    for rental in customer["rentals"]:
        genre = rental["movie"]["genre"]
        revenue_doc[genre] = revenue_doc.get(genre, 0) + rental["price"]

print("\n3) Total Revenue Per Genre")
print(revenue_doc)


3) Total Revenue Per Genre
{'Thriller': 47.78, 'Sci-Fi': 14.78, 'Comedy': 43.120000000000005, 'Drama': 15.620000000000001, 'Romance': 29.63, 'Animation': 20.67}


In [47]:
eligible_doc = [customer["name"]
    for customer in customer_documents
    if all(rental["movie"]["rating"] >= 3 for rental in customer["rentals"])
]
print("\n4) Customers Who Never Rented Movie Rated Below 3")
print(eligible_doc)


4) Customers Who Never Rented Movie Rated Below 3
['Peter Gill', 'Christopher Hughes', 'Jamie Hill', 'Jose Mccullough']


In the document model, retrieving one customer's full rental history is easier because all related data is already nested inside the customer document. No joins are required. However, aggregation queries such as:
- top 5 most-rented movies
- total revenue per genre
are harder because we must manually loop through every customer and every rental entry.
In the relational model, aggregation queries are simpler and cleaner using groupby operations, while customer-specific reads require joins. The document model has a clear advantage for user-centered queries (read one customer with all related data). The relational model has a clear advantage for analytical and cross-customer aggregation queries.


In [48]:
nodes = {
    "c001": {"type": "customer", "name": "Alice"},
    "c002": {"type": "customer", "name": "Bob"},
    "c003": {"type": "customer", "name": "Charlie"},
    "c004": {"type": "customer", "name": "Diana"},

    "m001": {"type": "movie", "title": "Inception"},
    "m002": {"type": "movie", "title": "The Dark Knight"},
    "m003": {"type": "movie", "title": "Interstellar"},
    "m004": {"type": "movie", "title": "Titanic"},
    "m005": {"type": "movie", "title": "The Matrix"},

    "Sci-Fi": {"type": "genre"},
    "Action": {"type": "genre"},
    "Drama": {"type": "genre"},

    "Baku": {"type": "city"},
    "Ganja": {"type": "city"},
}

In [49]:
edges = [
    ("c001", "rented", "m001"),
    ("c001", "rented", "m002"),
    ("c001", "rented", "m003"),

    ("c002", "rented", "m002"),
    ("c002", "rented", "m004"),

    ("c003", "rented", "m001"),
    ("c003", "rented", "m005"),

    ("c004", "rented", "m004"),

    ("m001", "belongs_to", "Sci-Fi"),
    ("m002", "belongs_to", "Action"),
    ("m003", "belongs_to", "Sci-Fi"),
    ("m004", "belongs_to", "Drama"),
    ("m005", "belongs_to", "Sci-Fi"),

    ("c001", "lives_in", "Baku"),
    ("c002", "lives_in", "Ganja"),
    ("c003", "lives_in", "Baku"),
    ("c004", "lives_in", "Ganja"),
]

In [50]:
def movies_rented_by(customer_id):
    rented_movies = [
        target for source, relation, target in edges
        if source == customer_id and relation == "rented"
    ]
    return [nodes[m]["title"] for m in rented_movies]

print(movies_rented_by("c001"))

['Inception', 'The Dark Knight', 'Interstellar']


In [51]:
from collections import Counter

def preferred_genres(customer_id):
    movies = [
        target for source, relation, target in edges
        if source == customer_id and relation == "rented"
    ]

    genres = []
    for movie in movies:
        for source, relation, target in edges:
            if source == movie and relation == "belongs_to":
                genres.append(target)

    return Counter(genres)

print(preferred_genres("c001"))

Counter({'Sci-Fi': 2, 'Action': 1})


In [52]:
def similar_customers(customer_id):
    target_genres = preferred_genres(customer_id).keys()
    similar = set()

    for source, relation, target in edges:
        if relation == "rented" and source != customer_id:
            # Check genre of rented movie
            for s2, r2, g in edges:
                if s2 == target and r2 == "belongs_to" and g in target_genres:
                    similar.add(source)

    return list(similar)

print(similar_customers("c001"))

['c002', 'c003']


In [53]:
def recommend_movies(customer_id):
    rented = {
        target for source, relation, target in edges
        if source == customer_id and relation == "rented"
    }

    genres = preferred_genres(customer_id).keys()

    recommendations = []

    for source, relation, target in edges:
        if relation == "belongs_to" and target in genres:
            if source not in rented:
                recommendations.append(nodes[source]["title"])

    return recommendations

print(recommend_movies("c001"))

['The Matrix']


### Why Recommendation is Hard in SQL or Documents

The recommendation query requires multi-hop traversal:
Customer → Movies → Genres → Other Movies (not yet rented).

In SQL, this requires multiple JOIN operations and subqueries, which become complex and harder to maintain as relationships grow. The logic becomes less intuitive because relational databases are optimized for set-based operations, not deep graph traversal.

In the document model, the issue is duplication. Genre information is embedded in many customer documents. To find recommendations, we must scan all documents, extract nested arrays, and perform manual filtering logic.

In the graph model, traversal is natural. We simply "walk" along edges. Each step corresponds directly to a real-world relationship. That makes graph databases ideal for recommendation systems and relationship-heavy queries.

In [54]:
import pandas as pd
import numpy as np

np.random.seed(42)

transactions = pd.DataFrame({
    "transaction_id": [f"t{i:05d}" for i in range(5000)],
    "customer_id": np.random.randint(1, 500, 5000),
    "product_id": np.random.randint(1, 200, 5000),
    "amount": np.round(np.random.uniform(5, 500, 5000), 2),
    "payment_method": np.random.choice(["card", "cash", "paypal"], 5000),
    "timestamp": pd.date_range("2025-01-01", periods=5000, freq="h"),
    "status": np.random.choice(["completed", "refunded", "pending"], 5000)
})

In [55]:
transactions.loc[transactions["transaction_id"] == "t00010"]

new_transaction = pd.DataFrame([{
    "transaction_id": "t99999",
    "customer_id": 101,
    "product_id": 50,
    "amount": 120.00,
    "payment_method": "card",
    "timestamp": pd.Timestamp.now(),
    "status": "pending"
}])

transactions = pd.concat([transactions, new_transaction], ignore_index=True)

transactions.loc[transactions["transaction_id"] == "t99999", "status"] = "completed"

allowed_status = {"completed", "refunded", "pending"}

assert (transactions["amount"] > 0).all()
assert set(transactions["status"]).issubset(allowed_status)

In [56]:
transactions.groupby("payment_method")["amount"].sum()

transactions["month"] = transactions["timestamp"].dt.to_period("M")
transactions.groupby("month")["amount"].mean()

transactions.groupby("customer_id")["amount"].sum().sort_values(ascending=False).head(10)

refund_rate = (
    (transactions["status"] == "refunded").sum()
    / len(transactions)
) * 100

refund_rate

np.float64(34.17316536692662)

# Revenue by payment method
transactions.groupby("payment_method")["amount"].sum()

# Average amount by month
transactions["month"] = transactions["timestamp"].dt.to_period("M")
transactions.groupby("month")["amount"].mean()

# Top 10 customers
transactions.groupby("customer_id")["amount"].sum().sort_values(ascending=False).head(10)

# Refund rate
refund_rate = (
    (transactions["status"] == "refunded").sum()
    / len(transactions)
) * 100

refund_rate

# Task 5: Choosing the Right Model

## 1. Hospital Patient Record System

**Recommended Model:** Document  
**Processing Type:** OLTP  

A hospital patient record system stores complex and hierarchical information such as visits, prescriptions, lab tests, diagnoses, and doctor notes. All of this data belongs naturally to a single patient entity. A document model is suitable because each patient can be stored as one structured document containing nested records. This allows doctors to retrieve a complete medical history quickly without performing multiple joins.

The system mainly performs frequent reads and updates (adding test results, updating prescriptions), which makes it transactional in nature. Therefore, OLTP is the correct processing paradigm because it focuses on fast, consistent operations on individual records.

---

## 2. Social Network with Friend Suggestions

**Recommended Model:** Graph  
**Processing Type:** OLTP  

A social network requires relationship-based queries such as "friends of friends" and connection recommendations. These queries involve multi-hop traversal across user relationships. A graph model is ideal because users are nodes and friendships are edges, making traversal natural and efficient.

Since users expect real-time suggestions and instant updates when they add or remove friends, the system must handle frequent small transactions. Therefore, this workload fits OLTP, as it prioritizes quick lookups and relationship updates.

---

## 3. Financial Reporting System

**Recommended Model:** Relational  
**Processing Type:** OLAP  

A financial reporting system computes monthly revenue summaries, regional breakdowns, and product category analytics. This type of data is structured and tabular, which fits naturally into a relational model with well-defined schemas.

The system performs heavy aggregation queries over large datasets rather than frequent single-row updates. These operations are analytical in nature, making OLAP the correct processing paradigm. OLAP systems are optimized for scanning large volumes of data and computing summaries efficiently.

---

## 4. IoT Real-Time Sensor Platform

**Recommended Model:** Relational (or Time-Series Database)  
**Processing Type:** Hybrid (OLTP + Analytical Processing)  

An IoT platform receives continuous data from thousands of devices. The system must handle high-frequency inserts, which makes it transactional in nature. At the same time, it must analyze sensor readings over time to detect anomalies.

Because it combines heavy write operations with analytical processing, the workload is hybrid. The ingestion of sensor data behaves like OLTP, while anomaly detection and trend analysis resemble OLAP-style operations.

---

## 5. Content Management System

**Recommended Model:** Document  
**Processing Type:** OLTP  

A content management system stores articles where each article may have different metadata fields such as tags, categories, SEO information, images, and custom attributes. Since the structure may vary between articles, a document model provides schema flexibility.

The system primarily performs create, read, update, and delete operations on individual articles. These operations are transactional and require consistency, making OLTP the appropriate processing type.